# EPUB Text Extractor for Text-to-Speech (TTS)

This notebook allows you to upload an EPUB file, extract its text content, and clean it for Text-to-Speech (TTS) applications. The goal is to get a clean, readable version of the EPUB's main content.

**Steps:**
1.  Ensure all prerequisite libraries are installed.
2.  Run the cells sequentially.
3.  Upload your EPUB file using the widget.
4.  The notebook will extract and clean the text.
5.  The final text will be available in the `processed_epub_text` variable.

## Prerequisites and Installation

This notebook requires the following Python libraries:

*   `EbookLib`: For reading and parsing EPUB files.
*   `BeautifulSoup4`: For parsing HTML content within the EPUB.
*   `ipywidgets`: For the interactive file upload widget in Jupyter.
*   `notebook`: The Jupyter Notebook environment itself.

You can install these libraries using pip. Run the following command in your terminal or a code cell in Jupyter (uncomment it first):

```python
# !pip install EbookLib BeautifulSoup4 ipywidgets notebook
```

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

upload_widget_epub = FileUpload(accept='.epub', multiple=False)
display(upload_widget_epub)

The following code cell will extract text from the uploaded EPUB file. It uses the `EbookLib` library to parse the EPUB structure and `BeautifulSoup4` to extract text content from the HTML files within the EPUB.

In [ ]:
import io
from ebooklib import epub
from bs4 import BeautifulSoup
import ebooklib # Required for ebooklib.ITEM_DOCUMENT

def extract_text_from_epub(epub_upload_value):
    """Extracts text from an uploaded EPUB file."""
    if not epub_upload_value:
        print("No EPUB file uploaded.")
        return None

    file_content = epub_upload_value[0]['content']
    epub_file_like_object = io.BytesIO(file_content)
    all_text_parts = []

    try:
        book = epub.read_epub(epub_file_like_object)
        for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
            html_content = item.get_content()
            soup = BeautifulSoup(html_content, 'html.parser')
            text = soup.get_text(separator='\n')
            all_text_parts.append(text)
        
        full_text = "\n\n".join(all_text_parts)
        print("Successfully extracted text from EPUB.")
        return full_text
    except Exception as e:
        print(f"Error processing EPUB: {e}")
        return None

# --- Example usage (illustrative, run this cell after uploading an EPUB) ---
# raw_epub_text = None # Initialize to prevent potential NameError
# if upload_widget_epub.value:
#     raw_epub_text = extract_text_from_epub(upload_widget_epub.value)
#     if raw_epub_text:
#         print("Sample of extracted EPUB text (first 500 chars):\n", raw_epub_text[:500])
#     else:
#         print("Failed to extract text or EPUB was empty.")
# else:
#     print("Please upload an EPUB file first using the widget above.")


The following Python code will perform text cleaning on the extracted EPUB content. This involves normalizing newlines and spaces to create a clean, readable text block suitable for Text-to-Speech (TTS) applications. The aim is to preserve paragraph structure while removing unnecessary formatting.

In [ ]:
import re

def clean_epub_text(raw_text):
    """Cleans the raw extracted EPUB text."""
    if not raw_text or not isinstance(raw_text, str):
        print("Warning: No text to clean or input is not a string. Returning empty string.")
        return ""

    text = raw_text
    # 1. Normalize newlines
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # 2. Collapse excessive newlines (3 or more) to double newlines (paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 3. Replace single newlines between words (alphanumeric characters) with a single space
    # This helps join lines that were part of the same sentence but broken by a single newline.
    text = re.sub(r'(\w)\n(\w)', r'\1 \2', text)

    # 4. Replace multiple spaces/tabs with a single space
    text = re.sub(r'[ \t]{2,}', ' ', text)

    # 5. Strip leading/trailing whitespace from the whole text
    cleaned_text = text.strip()

    return cleaned_text

# --- Example usage (illustrative) ---
# # Assuming 'raw_epub_text' variable holds the text from the EPUB extraction cell.
# # If that cell hasn't been run or if no file was uploaded, raw_epub_text might be None or not defined.

# processed_epub_text = "" # Initialize to prevent NameError
# if 'raw_epub_text' in locals() and raw_epub_text:
#     processed_epub_text = clean_epub_text(raw_epub_text)
#     print("Cleaned EPUB text ready for TTS. Sample (first 500 chars):\n", processed_epub_text[:500])
# elif 'raw_epub_text' in locals():
#     print("Raw EPUB text is empty. Cleaning resulted in empty text.")
# else:
#     print("Variable 'raw_epub_text' not found. Run the EPUB extraction cell first.")

# # For quick testing, you can use a sample string:
# sample_raw_epub = "Chapter 1\n\n\nThis is the first sentence.\nThis sentence continues on a new line.\n\n   This is another paragraph with  too much  spacing and\r\na final line."
# print(f"Original Sample:\n'{sample_raw_epub}'\n")
# cleaned_sample = clean_epub_text(sample_raw_epub)
# print(f"Cleaned Sample:\n'{cleaned_sample}'")


## How to Use and Final Output

1.  **Run Cells Sequentially**: Execute each code cell in this notebook from top to bottom.
2.  **Upload EPUB**: When you run the cell with the "Upload EPUB" button, click it and select your `.epub` file.
3.  **Text Extraction and Cleaning**: The subsequent cells will automatically process the uploaded file, extracting text using `EbookLib` and `BeautifulSoup4`, and then cleaning it.
4.  **Final Processed Text**: The cleaned text is stored in a Python variable named `processed_epub_text`. A sample of this text will be printed by the last code cell. This variable now holds the text formatted for use with a TTS system.

**Using with a TTS Engine:**
The `processed_epub_text` can now be passed to a Text-to-Speech library or API of your choice (e.g., gTTS, pyttsx3, Google Cloud TTS). This notebook focuses on preparing the text from EPUB files for such systems.